<a href="https://colab.research.google.com/github/erdijova/scikit-learn-cookbook/blob/main/2_Pre_Model_Workflow_and_Data_Preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### The impact of raw data on model performance

ML algorithms are designed to learn patterns from data. However, when the input data is flawed—
whether due to missing data, outliers, or irrelevant features—the model’s ability to generalize from
training data on unseen data diminishes. For instance, a model trained on noisy or biased data may
yield inaccurate predictions, leading to poor decision-making in real-world applications.

### Common data issues

Some of the most common instances of data quality issues in ML model development include
the following:

• Missing data: Incomplete datasets are common (if not the standard) in real-world scenarios.
Missing data can arise from various sources, such as errors in data collection, system failures,
or data corruption. scikit-learn provides several strategies for handling missing data, including
imputation techniques using SimpleImputer() or KNNImputer().

• Outliers: Outliers can distort statistical analyses and lead to misleading results. Identifying
and treating outliers is vital for maintaining the integrity of your model. Techniques such as
Z-score analysis or using scalers such as RobustScaler() can help mitigate their impact.

• Categorical variables: Many ML algorithms require numerical input; thus, categorical
variables need to be transformed into a suitable format. scikit-learn offers utilities such as
OneHotEncoder() and LabelEncoder() to facilitate this transformation.

• Feature scaling: Features with different scales can negatively affect model convergence and
performance, especially for algorithms that rely on distance metrics (e.g., k-nearest neighbors
(KNN)). Standardization (StandardScaler()) and normalization (MinMaxScaler())
are common techniques used to scale features appropriately.

• Data leakage: This occurs when information from outside the training dataset is used to create
the model, leading to overly optimistic performance metrics. Careful separation of training
and testing datasets during preprocessing is essential to prevent this issue. This is a common
challenge for those in the MLOps field, where managing real-time data streams is central to
mitigating this occurrence.

**Getting ready**

To begin, we will create a toy dataset composed of random, quantitative data, 10 features, and
several missing data values randomly spread throughout. We will then store the dataset in a pandas
DataFrame() object for better readability.

In [1]:
import numpy as np
import pandas as pd

In [2]:
np.random.seed(2024) # For reproducibility
n_samples = 20
n_features = 10

In [3]:
data = {
f"Feature{i+1}": np.random.uniform(0, 100, n_samples)
for i in range(n_features)
}

In [9]:
df = pd.DataFrame(data)

In [14]:
for column in df.columns:
    mask = np.random.random(n_samples) < 0.2
    df.loc[mask, column] = np.nan

In [15]:
display(df)

,Feature1,Feature2,Feature3,Feature4,Feature5,Feature6,Feature7,Feature8,Feature9,Feature10
0,58.801452,NaN,42.009814,34.680397,49.962259,NaN,NaN,89.954588,41.152421,NaN
1,69.910875,9.554215,6.436369,31.287816,37.966499,NaN,82.005649,NaN,75.797616,91.382492
2,NaN,96.090974,59.643269,84.710402,NaN,84.109027,NaN,70.288128,1.778343,4.117690
3,NaN,25.176729,83.732372,88.023110,16.886931,97.205554,84.696823,NaN,NaN,80.077973
4,20.501895,NaN,89.248639,67.655865,58.635861,78.225721,60.911562,NaN,65.114243,99.119187
5,10.606287,76.825393,20.052744,5.367515,NaN,19.703051,34.423301,93.399494,72.206680,12.640276
6,72.724014,79.792340,50.239523,55.921377,6.191019,NaN,22.966899,21.049022,57.358544,14.302591
7,NaN,NaN,89.538184,69.451294,NaN,47.885551,NaN,33.620401,99.685711,6.683138
8,47.384570,NaN,25.592093,82.419730,73.414540,61.663700,29.172571,65.946718,61.005155,34.052747
9,44.829582,38.165095,NaN,31.142866,28.865545,NaN,41.004459,41.460336,50.236236,NaN


**How to do it...**

There are a variety of methods for dealing with missing data. scikit-learn contains three approaches:
SimpleImputer(), KNNImputer(), and IterativeImputer(). They are outlined next.

**Using the SimpleImputer() class**

The SimpleImputer() class in scikit-learn is one of the most straightforward methods for handling
missing values. It allows users to replace missing entries with a specified statistic such as the mean,
median, or most frequent value. This method is particularly useful when dealing with numerical features.

In [16]:
from sklearn.impute import SimpleImputer

In [17]:
imputer = SimpleImputer(strategy="mean")

In [18]:
imputed_data = imputer.fit_transform(df)
imputed_df = pd.DataFrame(imputed_data, columns=df.columns)
imputed_df

,Feature1,Feature2,Feature3,Feature4,Feature5,Feature6,Feature7,Feature8,Feature9,Feature10
0,58.801452,53.864661,42.009814,34.680397,49.962259,57.822964,51.145011,89.954588,41.152421,48.400044
1,69.910875,9.554215,6.436369,31.287816,37.966499,57.822964,82.005649,50.715003,75.797616,91.382492
2,52.558605,96.090974,59.643269,84.710402,46.055883,84.109027,51.145011,70.288128,1.778343,4.117690
3,52.558605,25.176729,83.732372,88.023110,16.886931,97.205554,84.696823,50.715003,60.616723,80.077973
4,20.501895,53.864661,89.248639,67.655865,58.635861,78.225721,60.911562,50.715003,65.114243,99.119187
5,10.606287,76.825393,20.052744,5.367515,46.055883,19.703051,34.423301,93.399494,72.206680,12.640276
6,72.724014,79.792340,50.239523,55.921377,6.191019,57.822964,22.966899,21.049022,57.358544,14.302591
7,52.558605,53.864661,89.538184,69.451294,46.055883,47.885551,51.145011,33.620401,99.685711,6.683138
8,47.384570,53.864661,25.592093,82.419730,73.414540,61.663700,29.172571,65.946718,61.005155,34.052747
9,44.829582,38.165095,52.897507,31.142866,28.865545,57.822964,41.004459,41.460336,50.236236,48.400044


After application, we can see the results of SimpleImputer(), for example, on Feature1. The
original dataset had both the second and third records as NaN, but these have been replaced by the
calculated mean value (i.e., 52.558605) for that feature.

**Using the KNNImputer() class**

For more complex datasets, KNNImputer() can be employed. This method uses the KNN algorithm
to impute missing values based on the values of neighboring samples. Simply put, KNNImputer()
takes each missing value and identifies the nearest labeled values in the dataset’s feature space (the
number of labeled values is determined by the 'n_neighbors' hyperparameter). Then, based
on the majority label appearing among those values, the missing value is assigned the same. This
technique can capture more nuanced relationships within the data compared to simpler methods.

In [19]:
from sklearn.impute import KNNImputer

In [20]:
knn_imputer = KNNImputer(n_neighbors=2)

In [21]:
knn_imputed_data = knn_imputer.fit_transform(df)
knn_imputed_df = pd.DataFrame(
knn_imputed_data, columns=df.columns)
knn_imputed_df

,Feature1,Feature2,Feature3,Feature4,Feature5,Feature6,Feature7,Feature8,Feature9,Feature10
0,58.801452,48.954043,42.009814,34.680397,49.962259,93.910386,52.549271,89.954588,41.152421,59.833678
1,69.910875,9.554215,6.436369,31.287816,37.966499,86.793468,82.005649,45.416191,75.797616,91.382492
2,67.752344,96.090974,59.643269,84.710402,73.338309,84.109027,59.012876,70.288128,1.778343,4.117690
3,47.880864,25.176729,83.732372,88.023110,16.886931,97.205554,84.696823,64.510281,39.726833,80.077973
4,20.501895,31.670912,89.248639,67.655865,58.635861,78.225721,60.911562,25.903612,65.114243,99.119187
5,10.606287,76.825393,20.052744,5.367515,59.716773,19.703051,34.423301,93.399494,72.206680,12.640276
6,72.724014,79.792340,50.239523,55.921377,6.191019,40.482911,22.966899,21.049022,57.358544,14.302591
7,47.629715,77.843853,89.538184,69.451294,32.753328,47.885551,55.683434,33.620401,99.685711,6.683138
8,47.384570,35.180639,25.592093,82.419730,73.414540,61.663700,29.172571,65.946718,61.005155,34.052747
9,44.829582,38.165095,29.188271,31.142866,28.865545,59.674651,41.004459,41.460336,50.236236,35.000820


**Using the IterativeImputer() class**

IterativeImputer() offers an advanced approach by modeling each feature with missing
values as a function of other features in a round-robin fashion. More explicitly, this method models
each feature with missing values as a regression function, where it takes the place of the y value (the
output), and all other features (without missing values) take the place of x values (the input). Then
the regression model is used to predict the missing values for the feature. This process is repeated for
each feature with missing values. This method can provide more accurate imputations by considering
multivariate relationships in the dataset.

In [22]:
from sklearn.experimental import (
enable_iterative_imputer
)# Experimental feature requires loading
from sklearn.impute import IterativeImputer

In [23]:
iterative_imputer = IterativeImputer()

In [24]:
iterative_imputed_data = iterative_imputer.fit_transform(df)
iterative_imputed_df = pd.DataFrame(
iterative_imputed_data, columns=df.columns)
iterative_imputed_df

,Feature1,Feature2,Feature3,Feature4,Feature5,Feature6,Feature7,Feature8,Feature9,Feature10
0,58.801452,54.365008,42.009814,34.680397,49.962259,58.680582,51.178102,89.954588,41.152421,48.292275
1,69.910875,9.554215,6.436369,31.287816,37.966499,60.070634,82.005649,50.710323,75.797616,91.382492
2,52.539676,96.090974,59.643269,84.710402,46.121290,84.109027,51.155371,70.288128,1.778343,4.117690
3,52.630591,25.176729,83.732372,88.023110,16.886931,97.205554,84.696823,50.680265,50.650877,80.077973
4,20.501895,53.198954,89.248639,67.655865,58.635861,78.225721,60.911562,50.691674,65.114243,99.119187
5,10.606287,76.825393,20.052744,5.367515,46.156310,19.703051,34.423301,93.399494,72.206680,12.640276
6,72.724014,79.792340,50.239523,55.921377,6.191019,55.923578,22.966899,21.049022,57.358544,14.302591
7,52.558131,53.942885,89.538184,69.451294,46.070477,47.885551,50.969808,33.620401,99.685711,6.683138
8,47.384570,54.204978,25.592093,82.419730,73.414540,61.663700,29.172571,65.946718,61.005155,34.052747
9,44.829582,38.165095,52.882614,31.142866,28.865545,55.978300,41.004459,41.460336,50.236236,48.429901


This imputation technique is a bit more computationally complex, but should elicit slightly better
predictions for missing values.

**Scaling techniques**

When working with datasets, features can have vastly different scales. For instance, a feature representing
age may range from 0 to 100, while another feature representing income could range from 0 to 100,000.
Many ML algorithms, such as KNN and gradient descent-based methods (e.g., linear regression), are
sensitive to these differences in scale. Therefore, scaling helps ensure that no single feature dominates
the learning process. This recipe covers the three most commonly used scaling techniques in ML.

**StandardScaler()**

Standardization transforms features to have a mean of 0 and a standard deviation of 1. This method
is particularly useful when the data follows a Gaussian (normal) distribution.

In [25]:
from sklearn.preprocessing import StandardScaler

In [26]:
scaler = StandardScaler()

In [27]:
scaled_data = scaler.fit_transform(iterative_imputed_df)
scaled_df = pd.DataFrame(
scaled_data, columns= iterative_imputed_df.columns)
scaled_df

,Feature1,Feature2,Feature3,Feature4,Feature5,Feature6,Feature7,Feature8,Feature9,Feature10
0,0.271931,0.019495,-0.373057,-0.894258,0.188761,0.031844,0.001905,1.683800,-0.780898,-0.003574
1,0.756048,-1.874939,-1.592196,-1.043864,-0.391744,0.085949,1.426871,-0.000534,0.652484,1.429056
2,-0.000939,1.783513,0.231259,1.311954,0.002887,1.021588,0.000854,0.839730,-2.409929,-1.472256
3,0.003022,-1.214477,1.056818,1.458037,-1.411839,1.531340,1.551267,-0.001824,-0.387917,1.053212
4,-1.397054,-0.029802,1.245866,0.559887,0.608499,0.792594,0.451822,-0.001334,0.210478,1.686279
5,-1.828276,0.969036,-1.125549,-2.186891,0.004582,-1.485267,-0.772566,1.831653,0.503915,-1.188904
6,0.878636,1.094467,-0.091017,0.042422,-1.929442,-0.075466,-1.302124,-1.273574,-0.110399,-1.133636
7,-0.000135,0.001649,1.255789,0.639061,0.000428,-0.388327,-0.007723,-0.734020,1.640810,-1.386962
8,-0.225584,0.012729,-0.935710,1.210940,1.323678,0.147955,-1.015275,0.653400,0.040472,-0.476999
9,-0.336923,-0.665377,-0.000435,-1.050256,-0.832163,-0.073336,-0.468360,-0.397536,-0.405072,0.001002


**MinMaxScaler()**

Min-max scaling rescales features to a fixed range, typically [0, 1]. This technique is useful when you
want to preserve the relationships between values while ensuring that all features contribute equally.

In [28]:
from sklearn.preprocessing import MinMaxScaler

In [29]:
minmax_scaler = MinMaxScaler()

In [30]:
minmax_scaled_data = minmax_scaler.fit_transform(
iterative_imputed_df)
minmax_scaled_df = pd.DataFrame(
minmax_scaled_data,
columns=iterative_imputed_df.columns
)
minmax_scaled_df

,Feature1,Feature2,Feature3,Feature4,Feature5,Feature6,Feature7,Feature8,Feature9,Feature10
0,0.603506,0.517824,0.380540,0.354639,0.569020,0.555896,0.435006,0.962767,0.402156,0.464988
1,0.721357,0.000000,0.000369,0.313594,0.413077,0.571920,0.833074,0.538604,0.756013,0.918562
2,0.537080,1.000000,0.568987,0.959922,0.519088,0.849027,0.434713,0.750206,0.000000,0.000000
3,0.538045,0.180530,0.826426,1.000000,0.139045,1.000000,0.867825,0.538279,0.499171,0.799569
4,0.197218,0.504349,0.885377,0.753589,0.681776,0.781206,0.560692,0.538402,0.646896,1.000000
5,0.092244,0.777371,0.145886,0.000000,0.519543,0.106575,0.218656,1.000000,0.719336,0.089710
6,0.751199,0.811657,0.468490,0.611621,0.000000,0.524114,0.070723,0.218016,0.567681,0.107208
7,0.537276,0.512946,0.888472,0.775311,0.518428,0.431454,0.432317,0.353891,1.000000,0.027004
8,0.482394,0.515975,0.205085,0.932208,0.873897,0.590284,0.150855,0.703283,0.604927,0.315101
9,0.455290,0.330621,0.496737,0.311840,0.294766,0.524745,0.303637,0.438627,0.494936,0.466437


**Normalizer()**

Normalization (L1 or L2 normalization) scales individual samples to have the unit norm. This technique
is particularly useful when dealing with sparse data or when you want to treat each sample equally
regardless of its magnitude.

In [31]:
from sklearn.preprocessing import Normalizer

In [32]:
normalizer = Normalizer()

In [33]:
normalized_data = normalizer.fit_transform(
iterative_imputed_df)
normalized_df = pd.DataFrame(
normalized_data,
columns=iterative_imputed_df.columns
)
normalized_df

,Feature1,Feature2,Feature3,Feature4,Feature5,Feature6,Feature7,Feature8,Feature9,Feature10
0,0.339168,0.313579,0.242313,0.200037,0.288183,0.338471,0.295196,0.518860,0.237368,0.278551
1,0.376706,0.051482,0.034682,0.168591,0.204578,0.323683,0.441878,0.273247,0.408426,0.492404
2,0.264336,0.483450,0.300075,0.426192,0.232044,0.423166,0.257371,0.353631,0.008947,0.020717
3,0.243762,0.116607,0.387811,0.407684,0.078213,0.450213,0.392278,0.234729,0.234592,0.370886
4,0.095909,0.248868,0.417511,0.316499,0.274302,0.365945,0.284948,0.237139,0.304609,0.463686
5,0.068115,0.493382,0.128781,0.034471,0.296421,0.126535,0.221071,0.599823,0.463720,0.081177
6,0.460521,0.505281,0.318139,0.354119,0.039204,0.354133,0.145437,0.133292,0.363220,0.090570
7,0.274582,0.281816,0.467778,0.362837,0.240688,0.250170,0.266284,0.175644,0.520792,0.034915
8,0.265283,0.303467,0.143277,0.461427,0.411012,0.345225,0.163323,0.369203,0.341538,0.190645
9,0.321287,0.273524,0.379002,0.223196,0.206875,0.401188,0.293873,0.297140,0.360036,0.347090


Each technique provides a different approach to data scaling, but keep in mind that just like with
missing values, not all data requires data scaling. Again, tree-based methods such as decision trees and
Random Forest work using the raw data values for input features, so scaling these before modeling
would have a detrimental impact on results.

**Getting ready**

To begin, like we did earlier, we will create a toy dataset, only this time, our features will be composed
of qualitative data:

In [34]:
import numpy as np

In [35]:
np.random.seed(2024) # for reproducibility
categories = ["A", "B", "C", "D"]
categorical_data = pd.DataFrame(
{
"Department": np.random.choice(
categories, size=20),
"Position": np.random.choice(
["Junior", "Senior", "Manager"], size=20),
"Location": np.random.choice(
["NY", "SF", "LA", "CHI"], size=20),
}
)

In [36]:
display(categorical_data)

,Department,Position,Location
0,A,Manager,LA
1,C,Manager,LA
2,A,Junior,NY
3,A,Manager,LA
4,D,Manager,NY
5,A,Senior,LA
6,C,Manager,SF
7,D,Manager,LA
8,B,Junior,LA
9,D,Manager,CHI


OneHotEncoder()

One-hot encoding is a popular method for converting nominal categorical variables into a numerical
format. This technique creates binary columns for each category, allowing the model to treat each
category independently. For example, if a feature, Department, has four categories (A, B, C, and D),
one-hot encoding will create four new binary features and populate them with ones and zeros, where
a 1 indicates the original category for that particular record.

In [37]:
from sklearn.preprocessing import OneHotEncoder

In [38]:
onehot_encoder = OneHotEncoder(sparse_output=False)

In [39]:
onehot_encoded_data = onehot_encoder.fit_transform(
categorical_data)
onehot_encoded_df = pd.DataFrame(
onehot_encoded_data,
columns=onehot_encoder.get_feature_names_out()
)
onehot_encoded_df

,Department_A,Department_B,Department_C,Department_D,Position_Junior,Position_Manager,Position_Senior,Location_CHI,Location_LA,Location_NY,Location_SF
0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
1,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
2,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
3,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
4,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
5,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
6,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
7,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
8,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
9,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0


**Visualizing pipelines**

scikit-learn also allows you to visualize your pipeline using set_config() and display, which
can help in understanding the flow of data through various transformations.

In [40]:
from sklearn import set_config

In [41]:
set_config(display="diagram")

In [43]:
from sklearn.pipeline import Pipeline

pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

In [44]:
pipeline

Pipeline(steps=[('imputer', SimpleImputer()), ('scaler', StandardScaler())])